[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_v035_small_recurrent_ei.ipynb)

# jaxfne v0.3.5: Small Recurrent E/I Network

**Learning Objectives**

By the end of this tutorial, you will:
1. Build a small excitatory/inhibitory network (12–16 neurons) using the public Configuration grammar
2. Understand how recurrent connectivity between E and I populations emerges from neuron-type signatures
3. Observe how Emitter-state-based drive, recurrent coupling, and noise shape spike timing and voltage dynamics
4. Extract multimodal proxy readouts (spikes, voltage, source, LFP) from a single simulation
5. Recognize the computational scaffold scope: simulation is relative, not biophysically calibrated

## 2. Biological/Computational Question

**Question:** How does a small recurrent E/I population, driven by constant Emitter-state-based input and coupled through all-to-all signed weights, transform that drive into diverse voltage trajectories, spike rasters, and emergent temporal dynamics over 1 second?

**Computational framing:** In the Izhikevich reduced model, each neuron receives:
- A Emitter-state-based base drive (intrinsic parameter)
- Recurrent synaptic input from all other neurons (signed by their type)
- White noise (random perturbation)

We will simulate this for 12 neurons (9 excitatory, 3 inhibitory) with 0.1 ms timesteps and visualize the collective behavior.

## 3. Mathematical Glossary Flow

### Recurrent Input Equation

$$I_i(t) = I_i^{base} + \sum_j W_{ij} s_j(t) + \xi_i(t)$$

where:
- $I_i(t)$ = total input current to neuron $i$ at time $t$ (milliamperes, uncalibrated proxy units)
- $I_{drive} drive specific to cell type (excitatory: 5 pA proxy, inhibitory: 3 pA proxy)
- $W_{ij}$ = recurrent weight from neuron $j$ to neuron $i$ (dimensionless; positive if $j$ is excitatory, negative if $j$ is inhibitory)
- $s_j(t)$ = spike output of neuron $j$ (binary: 0 or 1 at each timestep)
- $\xi_i(t)$ = Gaussian white noise (standard deviation 0.5 proxy units)

### Worded Equation

"The total current driving neuron $i$ comes from three sources: a cell-type-dependent base drive, the weighted sum of recent spikes from all other neurons, and random noise."

### Implementation Location

In `jaxfne/emitters.py`, function `simulate_eig_izhikevich()` (lines 220–293):

```python
syn = weights @ prev_spikes  # Line 259 or 280
current_native = drive + syn + noise
```

The weights matrix `W` is generated at network construction time by `_default_eig_connectivity()` (lines 117–122) based on neuron-type signatures. Weights are all-to-all, excluding self-connections, scaled by $1/\sqrt{n}$ for stability.

### Boundary Statement

**This is a Emitter-state-based recurrent coupling scaffold, not a calibrated synaptic biophysics model.** Weights are phenomenological; no ion channels, neurotransmitter kinetics, or empirical synaptic parameters are included. The simulation demonstrates relative temporal dynamics, not a mechanistic proof.

## 4. Canonical Import

In [ ]:
import jaxfne as jtfne
import jax.numpy as jnp
import numpy as np
import json
from pathlib import Path

print(f"jaxfne {jtfne.__version__}")

## 5. Configuration Block

In [ ]:
def make_recurrent_ei_config():
    """Build a 12-neuron recurrent E/I configuration using chainable grammar."""
    cfg = jtfne.Configuration()
    cfg = cfg.runtime(
        seed=42,
        dtype="float32",
        duration_ms=1000.0,
        dt_ms=0.1,
    )
    cfg = cfg.column(
        "small_recurrent_ei",
        layers=["L2/3"],
        n=12,
    )
    cfg = cfg.cell_types({"E": 0.75, "PV": 0.25})  # 9 E, 3 I
    cfg = cfg.connectivity(
        excitatory_to_inhibitory=True,
        inhibitory_to_excitatory=True,
    )
    cfg = cfg.set_emitter("izhikevich", "cortical_eig_e_plus_pv")
    cfg = cfg.probes([
        "MUA-proxy",
        "source-proxy",
        "LFP-proxy",
        "CSD-proxy",
        "EEG-proxy",
        "MEG-proxy",
        "EMM-proxy",
    ])
    return cfg

cfg = make_recurrent_ei_config()
print(f"Configuration: {cfg.metadata['duration_ms']}ms simulation, dt={cfg.metadata['dt_ms']}ms")
print(f"Network: {cfg.networks[0]['n']} neurons")
print(f"Cell types: {cfg.networks[0]['cell_types']}")
print(f"Connectivity metadata: {cfg.metadata.get('connectivity', {})}")

## 6. Simulation Block

In [ ]:
# Construct the model from configuration
model = jtfne.construct(cfg)
print(f"Model constructed.")
print(f"  Network size: {len(model.params['emitter'].labels)} neurons")
print(f"  Neuron types: {model.params['emitter'].labels}")
print(f"  Weights shape: {model.params['emitter'].W.shape}")

# Run simulation
signals = jtfne.simulate(
    model,
    duration_ms=1000.0,
    dt_ms=0.1,
    seed=42,
)

print(f"\nSimulation complete.")
print(f"  Voltage: {signals.V_m.shape} (time × neurons)")
print(f"  Spikes: {signals.spikes.shape}")
print(f"  Source proxy: {signals.sources.shape if signals.sources is not None else 'None'}")

## 7. Probe/Readout Block

In [ ]:
# Extract probe metadata
probe_modes = cfg.probes[0]['modes'] if cfg.probes else []
print(f"Probe modes declared: {probe_modes}")
print(f"Probe operator status: {cfg.probes[0].get('operator_status', 'unknown')}")
print(f"Field solver status: {cfg.probes[0].get('field_solver_status', 'unknown')}")
print(f"Physical amplitude claim allowed: {cfg.probes[0].get('physical_amplitude_claim_allowed', 'unknown')}")

# Compute basic readouts from signals
n_steps = signals.V_m.shape[0]
n_neurons = signals.V_m.shape[1]
duration_ms = 1000.0

# Spike-derived readouts
spike_count_per_neuron = signals.spikes.sum(axis=0)
mean_firing_rate_hz = (spike_count_per_neuron.sum() / n_neurons / duration_ms) * 1000.0

# Voltage-derived readouts
mean_v_per_neuron = signals.V_m.mean(axis=0)
std_v_per_neuron = signals.V_m.std(axis=0)

print(f"\nBasic readout statistics:")
print(f"  Mean firing rate: {float(mean_firing_rate_hz):.2f} Hz")
print(f"  Spikes per neuron: {[int(c) for c in spike_count_per_neuron]}")
print(f"  Mean voltage (per neuron): {[f'{v:.1f}' for v in mean_v_per_neuron][:5]} ... (showing first 5)")
print(f"  Mean population voltage: {float(mean_v_per_neuron.mean()):.1f} mV")

## 8. Run Metadata and Scope Metadata

In [ ]:
# Collect run metadata
run_metadata = {
    "duration_ms": 1000.0,
    "dt_ms": 0.1,
    "n_steps": int(n_steps),
    "n_neurons": int(n_neurons),
    "seed": 42,
    "dtype": "float32",
}

# Scope metadata (scope boundaries)
scope_metadata = {
    "truth_mode": "truth_safe_unverified",
    "computational_level": "computational_scaffold",
    "physical_amplitude_claim_allowed": False,
    "field_solver_status": "laminar_proxy_no_pde",
    "source_calibration_status": "uncalibrated_izhikevich_proxy",
    "tutorial_role": "educational_demonstration",
}

# Network configuration metadata
network_metadata = {
    "n_total": int(n_neurons),
    "cell_type_labels": list(model.params['emitter'].labels),
    "n_excitatory": int(sum(1 for label in model.params['emitter'].labels if label == 'E')),
    "n_inhibitory": int(n_neurons - sum(1 for label in model.params['emitter'].labels if label == 'E')),
    "connectivity_type": "all_to_all_signed_by_type",
}

# Combine into validation report
validation_report = {
    "run_metadata": run_metadata,
    "scope_metadata": scope_metadata,
    "network_metadata": network_metadata,
    "signal_validity": {
        "voltage_finite": bool(jnp.isfinite(signals.V_m).all()),
        "spikes_finite": bool(jnp.isfinite(signals.spikes).all()),
        "voltage_range": [float(signals.V_m.min()), float(signals.V_m.max())],
        "mean_firing_rate_hz": float(mean_firing_rate_hz),
    },
}

# Verify JSON-safe (no NaN/Inf)
try:
    json.dumps(validation_report, allow_nan=False)
    print("✓ Validation report is JSON-safe (no NaN/Inf)")
except ValueError as e:
    print(f"✗ Validation report contains non-JSON-safe values: {e}")

print(f"\nValidation report (summary):")
print(f"  Truth mode: {scope_metadata['truth_mode']}")
print(f"  Computational level: {scope_metadata['computational_level']}")
print(f"  Network: {network_metadata['n_excitatory']} E + {network_metadata['n_inhibitory']} I neurons")
print(f"  Mean firing rate: {validation_report['signal_validity']['mean_firing_rate_hz']:.2f} Hz")
print(f"  Voltage finite: {validation_report['signal_validity']['voltage_finite']}")
print(f"  Spikes finite: {validation_report['signal_validity']['spikes_finite']}")

## 9. Figures

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Ensure output directory exists
output_dir = Path("tutorial_outputs/v035_small_recurrent_ei")
output_dir.mkdir(parents=True, exist_ok=True)
figures_dir = output_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# 1. Spike raster
fig, ax = plt.subplots(figsize=(12, 3))
spike_times, spike_units = np.where(np.array(signals.spikes))
time_ms = np.arange(signals.V_m.shape[0]) * 0.1
colors = ['red' if model.params['emitter'].labels[u] == 'E' else 'blue' for u in spike_units]
ax.scatter(time_ms[spike_times], spike_units, c=colors, s=10, alpha=0.6)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Neuron index')
ax.set_title('v0.3.5: Recurrent E/I Spike Raster')
ax.set_ylim(-0.5, n_neurons - 0.5)
red_patch = mpatches.Patch(color='red', label='Excitatory')
blue_patch = mpatches.Patch(color='blue', label='Inhibitory')
ax.legend(handles=[red_patch, blue_patch])
plt.tight_layout()
fig.savefig(figures_dir / "01_recurrent_raster.png", dpi=100, bbox_inches='tight')
print("✓ Saved 01_recurrent_raster.png")
plt.close(fig)

In [ ]:
# 2. Voltage traces (subset of neurons)
fig, ax = plt.subplots(figsize=(12, 4))
time_ms = np.arange(signals.V_m.shape[0]) * 0.1
for i in range(min(6, n_neurons)):  # Show first 6 neurons
    color = 'red' if model.params['emitter'].labels[i] == 'E' else 'blue'
    ax.plot(time_ms, np.array(signals.V_m)[:, i], linewidth=0.5, alpha=0.7, color=color, label=f'N{i} ({model.params["emitter"].labels[i]})')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Voltage (mV)')
ax.set_title('v0.3.5: Voltage Trajectories (subset of neurons)')
ax.legend(fontsize=8, loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(figures_dir / "02_voltage_traces.png", dpi=100, bbox_inches='tight')
print("✓ Saved 02_voltage_traces.png")
plt.close(fig)

In [ ]:
# 3. Population firing rate (rolling average)
fig, ax = plt.subplots(figsize=(12, 3))
window_ms = 50  # 50 ms rolling window
window_steps = int(window_ms / 0.1)
firing_rate_raw = np.array(signals.spikes).sum(axis=1) / n_neurons * 1000.0 / 0.1  # Hz
if window_steps > 1:
    firing_rate_smoothed = np.convolve(firing_rate_raw, np.ones(window_steps) / window_steps, mode='same')
else:
    firing_rate_smoothed = firing_rate_raw
ax.plot(time_ms, firing_rate_smoothed, linewidth=1.0, color='black')
ax.fill_between(time_ms, firing_rate_smoothed, alpha=0.3, color='gray')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Population firing rate (Hz)')
ax.set_title(f'v0.3.5: Population Firing Rate ({window_ms}ms rolling avg)')
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(figures_dir / "03_population_rate.png", dpi=100, bbox_inches='tight')
print("✓ Saved 03_population_rate.png")
plt.close(fig)

In [ ]:
# 4. Source proxy activity
fig, ax = plt.subplots(figsize=(12, 3))
if signals.sources is not None:
    source_activity = np.array(signals.sources).sum(axis=1)
    ax.plot(time_ms, source_activity, linewidth=1.0, color='purple', label='Source proxy')
    ax.fill_between(time_ms, source_activity, alpha=0.3, color='purple')
else:
    # Proxy source from voltage dynamics
    current_proxy = np.array(signals.V_m).mean(axis=1)
    ax.plot(time_ms, current_proxy, linewidth=1.0, color='purple', label='Voltage proxy')
    ax.fill_between(time_ms, current_proxy, alpha=0.3, color='purple')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Source proxy (uncalibrated units)')
ax.set_title('v0.3.5: Source Proxy Activity')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(figures_dir / "04_source_proxy.png", dpi=100, bbox_inches='tight')
print("✓ Saved 04_source_proxy.png")
plt.close(fig)

In [ ]:
# 5. Readout summary (composite figure)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Top-left: Neuron-wise firing rate
ax = axes[0, 0]
rates = spike_count_per_neuron / 1000.0 * 1000  # Convert to Hz
colors_list = ['red' if model.params['emitter'].labels[i] == 'E' else 'blue' for i in range(n_neurons)]
ax.bar(range(n_neurons), np.array(rates), color=colors_list, alpha=0.7)
ax.set_xlabel('Neuron index')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title('Neuron-wise firing rate')
ax.grid(alpha=0.3, axis='y')

# Top-right: Voltage distribution
ax = axes[0, 1]
ax.hist(np.array(signals.V_m).flatten(), bins=50, color='gray', alpha=0.7, edgecolor='black')
ax.set_xlabel('Voltage (mV)')
ax.set_ylabel('Count')
ax.set_title('Voltage distribution')
ax.grid(alpha=0.3, axis='y')

# Bottom-left: E vs I spike count
ax = axes[1, 0]
e_spikes = sum(spike_count_per_neuron[i] for i in range(n_neurons) if model.params['emitter'].labels[i] == 'E')
i_spikes = sum(spike_count_per_neuron[i] for i in range(n_neurons) if model.params['emitter'].labels[i] != 'E')
ax.bar(['Excitatory', 'Inhibitory'], [e_spikes, i_spikes], color=['red', 'blue'], alpha=0.7)
ax.set_ylabel('Total spike count')
ax.set_title('E vs I spike totals')
ax.grid(alpha=0.3, axis='y')

# Bottom-right: Temporal statistics
ax = axes[1, 1]
ax.axis('off')
stats_text = f"""Simulation Summary
Duration: 1000 ms, dt: 0.1 ms
Neurons: {n_neurons} ({network_metadata['n_excitatory']} E, {network_metadata['n_inhibitory']} I)
Mean firing rate: {float(mean_firing_rate_hz):.2f} Hz
Voltage range: [{float(signals.V_m.min()):.1f}, {float(signals.V_m.max()):.1f}] mV
Connectivity: All-to-all, signed by type
Truth mode: truth_safe_unverified
Scope: Computational scaffold
"""
ax.text(0.1, 0.5, stats_text, fontsize=10, verticalalignment='center', family='monospace')

fig.suptitle('v0.3.5: Recurrent E/I Network Readout Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(figures_dir / "05_readout_summary.png", dpi=100, bbox_inches='tight')
print("✓ Saved 05_readout_summary.png")
plt.close(fig)

## 10. Interpretation

### What the network shows:

1. **E/I recurrent structure:** The network automatically couples excitatory neurons positively and inhibitory neurons negatively to all targets, creating a biologically-inspired signed weight architecture. This is NOT a calibrated model of real synapses, but a simplified demonstration.

2. **Voltage dynamics:** Each neuron's voltage trajectory reflects the integrated effect of Emitter-state-based drive, recurrent synaptic input, and noise. The spike threshold is fixed at 30 mV in the Izhikevich model.

3. **Firing patterns:** If the network fires, you will see clusters of spikes in the raster and intervals of quiescence. This emerges from the interplay between excitatory drive and inhibitory suppression—not from biological learning or plasticity.

4. **Population-level behavior:** The rolling firing rate curve reveals whether the population enters a regime of sustained spiking, bursting, or silence. The specific pattern depends on seed, initial conditions, and noise.

5. **Scope boundary:** This is a simulation using the Izhikevich reduced model, not a validated neuroscience finding. The network behavior is relative and demonstrative, not calibrated to any neural tissue.

## 11. Failure Modes

### Common observations in small recurrent networks:

1. **Silent network (no spikes):** May occur if Emitter-state-based drive is too low relative to inhibitory suppression. This is a valid exploration outcome, not a failure.

2. **All-or-nothing firing:** If noise or recurrent feedback is weak, the network might lock into a single attractor (all-spike or all-quiet).

3. **Instability (divergence):** Unlikely with the Izhikevich reset mechanism, but possible if weight magnitudes are very large or parameters are extreme.

4. **Numerical precision:** Simulations use float32 by design. Very long simulations (>10 seconds) may accumulate floating-point rounding, but are acceptable for this scope.

### Debugging checklist:
- Check that seed, drive, and inhibitory strength are set as intended in the configuration
- Verify that the simulation runs without errors (NaN/Inf checks pass)
- Visual inspection of voltage trajectories: peaks should reach ~30 mV and valleys ~–65 mV if model is working correctly
- Check firing rate: reasonable range is 0–25 Hz for this small network in this timescale

## 12. Exercises

1. **Vary the E/I split:** Change `cfg.cell_types({"E": 0.5, "PV": 0.5})` to test a balanced network. How does the firing pattern change?

2. **Increase network size:** Modify `n=12` to `n=50` and re-run. Do you observe different temporal structure?

3. **Change the emitter preset:** Try `"cortical_eig_e_plus_pv"` vs. generic `"cortical_eig"`. Which produces more spikes?

4. **Add external drive:** Modify the Emitter-state-based drive by creating a `drive_schedule` (see advanced tutorials for implementation).

5. **Inspect the recurrent weights:** Print `model.params['emitter'].W` to see the actual coupling matrix. Verify that it is all-to-all, signed by type, and has zero diagonal.

6. **Compare multiple seeds:** Run the same network with `seed=7`, `seed=42`, and `seed=99`. Are the rasters different? Are firing rates consistent?

7. **Export to JSON:** Save the full run metadata and signals as JSON and verify JSON-safe format using `json.dumps(..., allow_nan=False)`.

## 13. What This Tutorial Covers / Does Not Cover

### This tutorial COVERS:
✓ Building a small all-to-all recurrent E/I network using public jaxfne grammar  
✓ Observing how neuron-type signatures determine coupling polarity  
✓ Extracting multimodal proxy readouts (spikes, voltage, source, LFP, etc.)  
✓ Visualizing spike rasters, voltage trajectories, and population dynamics  
✓ Recognizing computational scaffold scope and truth-safe unverified status  
✓ Verifying JSON-safe output and finite-value constraints  

### This tutorial DOES NOT COVER:
✗ Biological calibration or validation of the Izhikevich model  
✗ Solved Maxwell equations or EEG/MEG forward models  
✗ Synaptic plasticity, learning, or adaptation  
✗ Multi-compartment neuron models or detailed ion-channel dynamics  
✗ Spatial field computation beyond laminar proxy operators  
✗ Optimization or parameter fitting  
✗ Whole-brain or multi-area simulation  

### Scope boundary statement:
**jaxfne v0.3.5 is a computational scaffold for source-field-probe-readout workflows.** It does not claim to simulate biological tissue, produce calibrated neural measurements, or prove neuroscience hypotheses. All outputs are labeled `truth_safe_unverified` and `computational_scaffold`. Use this tutorial to understand the jaxfne pipeline and API, not as evidence for neural mechanism.